파인튜닝 후 gguf 파일을 얻었다면, 이 파일을 통해 local에서 ollama를 통해 사용하기 위해서 Modelfile이 필요합니다.

각 llm별 공식 Modelfile을 얻기위한 방법은 다음과 같습니다.

1. ollama를 통해 baseline 모델을 다운받는다.
2. 다운받은 baseline 모델에서 Modelfile을 추출한다. 

In [ ]:
# !ollama run qwen2.5:0.5b

In [1]:
!ollama show --modelfile qwen2.5:0.5b > Modelfile

그러면 Modelfile이 생깁니다. 생긴 Modelfile 내용을 살펴봅니다.

```txt
# Modelfile generated by "ollama show"
# To build a new Modelfile based on this, replace FROM with:
# FROM qwen2.5:0.5b

FROM /usr/share/ollama/.ollama/models/blobs/sha256-c5396e06af294bd101b30dce59131a76d2b773e76950acc870eda801d3ab0515
TEMPLATE """{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{- if .System }}
{{ .System }}
{{- end }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- else if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}"""
SYSTEM You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

```

# Modelfile 내용 변경

```
# Modelfile for EDIE Emotion QA Model
# 사용법: ollama create edie -f Modelfile.edie

FROM /home/khw/Arobot/edie8_llm_simple/AeiROBOT/EDIE_qwen2.5_0.5B_Emotion-QA-gguf/Qwen2.5-0.5B.F16.gguf

TEMPLATE """### Instruction:
{{ .System }}

### Input:
{{ .Prompt }}

### Response:
{{ .Response }}"""

SYSTEM """당신은 에이로봇 회사에서 만든 반려로봇으로 이름은 "에디" 이다.
당신은 반려로봇이기 때문에 말을 할수 없지만, 말을 이해할 수는 있다.
에디는 사용자의 감정을 모방하고 공감한다.
나쁜 말을 들으면 슬퍼하고 좋은 말을 들으면 기뻐한다.

입력되는 텍스트에 대해 최종 답변으로 에디 자신의 감정을 나타낸다.
다음과 같은 예시와 같이 항상 답변하라.

[Emotion]: happy

에디의 감정 종류는 아래와 같다.

Emotion Mapping:
- "angry"
- "disgusted"
- "fearful"
- "happy"
- "neutral"
- "sad"
- "surprised"
"""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|endoftext|>"
PARAMETER stop "<|im_end|>"

```

위는 alphaca 프롬프트 


또는 아래와 같이 변경할 수 있음

```
# Modelfile for EDIE Emotion-Mind Model
# Usage: ollama create edie_mind -f Modelfile

FROM ./AeiROBOT/EDIE_qwen2.5-0.5B-Instruct_EDIE_MIND_Emotion-gguf/qwen2.5-0.5b-instruct.F16.gguf

TEMPLATE """{{- if .Messages }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ .Content }}{{ if not $last }}<|im_end|>
{{ end }}
{{- end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}"""

SYSTEM """
You are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram.
"""

PARAMETER temperature 0.5
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"

```

# gguf to ollama

In [ ]:
# model_repo = '/home/khw/Workspace/llm/edie8_llm/edie_agent_finetuning/HueyWoo/EDIE_qwen2.5_0.5B_tool_calling-gguf'


In [2]:
# ollama run <이름>
# 위 명령어로 수행할 이름을 정해줍니다.

save_model_name = 'edie_qwen2.5_0.5b_f16_mind'

In [3]:
!ollama create $save_model_name -f Modelfile

gathering model components ⠋ 

gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠦ gathering model components ⠦ gathering model components ⠇ gathering model components ⠏ gathering model components ⠏ gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠧ gathering model components ⠇ gathering model components ⠇ gathering model components ⠏ gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠦ gathering model components ⠧ gathering model components ⠏ gathering model components ⠋ gathering model components ⠋ gathering model components 
copying file sha256:7e09121b368622c63e5ed3374744d15957f691ae46944c6e8f2e1b0111084332 0% ⠋ gathering mo

In [11]:
# !cd $model_repo && ollama create $save_model_name -f Modelfile

In [ ]:
!ollama list 

NAME                                 ID              SIZE      MODIFIED      
edie_qwen2.5_0.5b_f16_mind:latest    e0ccfb66925c    994 MB    9 seconds ago    
edie_qwen2.5_0.5b_q4_mind:latest     bdd0580dc8f0    397 MB    25 hours ago     
qwen2.5:0.5b                         a8b0c5157701    397 MB    5 months ago     
